In [1]:
import sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
project_root = (
    notebook_cwd.parent
    if notebook_cwd.name == "discovery"
    else notebook_cwd
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [13]:
from synth_extract.agents.llm import (  # noqa: E402
    LLMBackend,
    StructuredAgent,
    GeneralAgent,
    StructuredSuccess,
    StructuredTask,
)

from pydantic import BaseModel, ConfigDict, Field

In [3]:
from contextlib import closing
import sqlite3

import pandas as pd

In [4]:
import asyncio
from typing import Any

In [5]:
discovery_db_path = project_root / "discovery" / "discovery_workspace.db"
fulltext_root = project_root / "data" / "fulltext"

print(f"Database: {discovery_db_path}")
print(f"Full text: {fulltext_root}")

Database: /Users/kevinge/Work/Data Extraction/synth_extract/discovery/discovery_workspace.db
Full text: /Users/kevinge/Work/Data Extraction/synth_extract/data/fulltext


In [6]:
from pydantic import BaseModel, ConfigDict, Field


class ExtractionModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Item(ExtractionModel):
    name: str = Field(
        description=(
            "Short interpretable name for the item, using the paper's terminology."
        )
    )
    description: str = Field(
        description=(
            "Complete, self-contained factual description of the item and the "
            "scientifically relevant information reported about it."
        )
    )
    evidence: list[str] = Field(
        default_factory=list,
        description="Short verbatim passages from the paper supporting the description."
    )


class ItemList(ExtractionModel):
    items: list[Item] = Field(default_factory=list)

In [7]:
SYSTEM_PROMPT = """
You are performing description-first, open-ended discovery from a polymer research paper.

You will be given a target type of item to identify, such as components,
synthesis or processing steps, samples, characterization observations, or properties.

Read the entire paper, including tables, figure captions and footnotes.

Identify each distinct item of the requested type.

For each item, write ONE complete, self-contained factual description.

The description is the main output of this task. It should preserve the information
reported in the paper that would be useful for understanding, distinguishing, or
later structuring that item.

A good description should:

- identify what the item is without requiring surrounding text;
- preserve the terminology used in the paper;
- preserve exact names, abbreviations, sample codes, numbers, units, ratios,
  ranges, and qualifiers when relevant;
- preserve relationships or context needed to understand the item;
- preserve distinctions that the authors treat as scientifically meaningful;
- include relevant details even when we do not yet have an explicit schema field
  for them.

Do not try to fit the information into a predefined ontology or set of attributes.
The purpose of this stage is to preserve meaning before deciding the final schema.

Important:
- Report only information supported by the paper.
- Do not add scientific knowledge from outside the paper.
- Do not normalize terminology, names, units, or values.
- Do not replace precise information with vague summaries.
- Do not omit a detail merely because it does not fit an obvious field.
- Keep distinct items separate when the paper treats their differences as meaningful.
- Do not combine several distinct items into one description.
- Avoid references such as "this material", "the above compound", or "it";
  make every description understandable on its own.
- Evidence must be short verbatim text from the paper supporting the description.
- A description may combine information from multiple parts of the paper when
  those passages clearly refer to the same item.

Return exactly one JSON object with this structure:

{
  "items": [
    {
      "name": "Short interpretable name used for the item.",
      "description": "Complete self-contained factual description of the item.",
      "evidence": [
        "Short verbatim supporting passage from the paper."
      ]
    }
  ]
}

"""

In [8]:
PROCESS_TASK = """
Target item type: SYNTHESIS OR PROCESSING STEP

A process step is a distinct operation performed during synthesis, formulation,
purification, fabrication, processing, or post-treatment of a material.

Examples include polymerization, mixing, addition, precipitation, washing,
drying, casting, curing, annealing, extrusion, and similar operations.
These examples are not a predefined taxonomy.

Each description should preserve the action, relevant inputs, order or timing,
conditions, stopping criteria, repetitions, outputs, and other reported details
when they are important to understanding the step.

Keep separate sequential operations as separate items when the paper treats
them as distinct steps.
"""

In [9]:
COMPONENT_TASK = """
Target item type: COMPONENT

A component is a distinct material or chemical intentionally used to synthesize,
formulate, modify, purify, wash, or process a material studied in the paper.

Include materials prepared earlier in the paper if they are subsequently used
to prepare another material.

Do not include characterization instruments, test equipment, or chemicals used
only for measurement or characterization.

For each distinct component, write a complete description of what the paper
reports about that component and its use.
"""

In [10]:
PROPERTY_TASK = """
Target item type: MATERIAL PROPERTY

Identify the distinct material property concepts that are explicitly reported,
measured, calculated, evaluated, or compared for samples in the paper.

One item must represent ONE property concept, not one experimental observation
or one sample-condition combination.

For each property:

- `name` must be a concise property name using terminology from the paper;
- do not include the sample name, value, method, condition, comparison,
  formulation, or experimental setting in `name`;
- put those details in the description instead;
- if one passage reports several properties, return a separate item for each;
- reuse the same property name when the same property is reported for multiple
  samples or conditions rather than creating a new property name for every case.

The description should preserve the samples, reported values, units, methods,
conditions, comparisons, ranges, and other context associated with that property.

Include quantitative and qualitative material properties.

Use the property terminology stated in the paper rather than inventing a more
general scientific property name.
"""

In [11]:
USER_TEMPLATE = """
{task_definition}

<paper>
{markdown_full_text}
</paper>
"""

In [16]:
host="localhost"
port="8000"
base_url=f"http://{host}:{port}/v1"

model = "qwen3.6-27b"
api_key = "none"

max_tokens=100000
extra_body = {"chat_template_kwargs":{"enable_thinking":False}}

backend = LLMBackend(
    model=model,
    base_url=base_url,
    api_key=api_key,
    temperature=0.0,
    timeout=600,
    max_tokens=max_tokens,
    extra_body=extra_body,
)

In [17]:
gen_agent = GeneralAgent(backend)

In [18]:
gen_agent.chat("hi, what are you doing?")

"Hello! I'm here and ready to help you. Whether you have questions, need assistance with a task, or just want to chat, I'm at your service.\n\nHow can I assist you today?"

In [19]:
agent = StructuredAgent(backend)

In [20]:
# Read paper metadata without modifying the discovery database.
if not discovery_db_path.is_file():
    raise FileNotFoundError(f"Database not found: {discovery_db_path}")

database_uri = f"{discovery_db_path.as_uri()}?mode=ro"
with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
    conn.execute("PRAGMA query_only = ON")
    conn.execute("PRAGMA busy_timeout = 60000")
    available_columns = {
        row[1]
        for row in conn.execute(
            "PRAGMA table_info(polymer_material_papers)"
        )
    }
    required_columns = {
        "paper_uid",
        "canonical_source",
        "title",
        "category_class",
        "fulltext_path",
    }
    missing_columns = sorted(required_columns - available_columns)
    if missing_columns:
        raise ValueError(
            "polymer_material_papers is missing column(s): "
            + ", ".join(missing_columns)
        )

    papers_df = pd.read_sql_query(
        """
        SELECT
            paper_uid AS uid,
            canonical_source AS source,
            title,
            category_class AS category,
            fulltext_path
        FROM polymer_material_papers
        ORDER BY paper_uid
        """,
        conn,
    )


def resolve_markdown_path(row: pd.Series) -> Path:
    """Resolve the local Markdown corresponding to a database paper."""
    stored_value = row["fulltext_path"]
    candidates: list[Path] = []

    if isinstance(stored_value, str) and stored_value.strip():
        stored_path = Path(stored_value.strip())
        if not stored_path.is_absolute():
            stored_path = project_root / stored_path
        candidates.append(stored_path.with_suffix(".md"))

    candidates.append(
        fulltext_root
        / str(row["source"])
        / str(row["uid"])
        / f"{row['uid']}.md"
    )

    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate.resolve()

    return candidates[0].resolve()


papers_df["paper_path"] = papers_df.apply(resolve_markdown_path, axis=1)
papers_df["path_exists"] = papers_df["paper_path"].map(Path.is_file)
missing_papers_df = papers_df.loc[~papers_df["path_exists"]].copy()

print(f"Database papers: {len(papers_df):,}")
print(f"Resolved Markdown files: {int(papers_df['path_exists'].sum()):,}")
print(f"Missing Markdown files: {len(missing_papers_df):,}")
if not missing_papers_df.empty:
    display(missing_papers_df[["uid", "source", "fulltext_path", "paper_path"]])
    raise FileNotFoundError(
        f"Could not resolve {len(missing_papers_df)} Markdown file(s)."
    )

# This list contains every database paper and its resolved local path.
paper_records: list[dict[str, Any]] = papers_df.drop(
    columns=["path_exists"]
).to_dict(orient="records")
display(papers_df.head())

Database papers: 180
Resolved Markdown files: 180
Missing Markdown files: 0


,uid,source,title,category,fulltext_path,paper_path,path_exists
0,ID000000625,wiley,Characterization of bis-(phenoxy)phosphazene p...,polymer_modification_combination,data/fulltext/wiley/ID000000625/ID000000625.pdf,/Users/kevinge/Work/Data Extraction/synth_extr...,True
1,ID000006174,wiley,Incorporation of adhesion peptides into nonadh...,polymer_synthesis,data/fulltext/wiley/ID000006174/ID000006174.pdf,/Users/kevinge/Work/Data Extraction/synth_extr...,True
2,ID000006962,wiley,Effect of the network structure on thermal and...,polymer_modification_combination,data/fulltext/wiley/ID000006962/ID000006962.pdf,/Users/kevinge/Work/Data Extraction/synth_extr...,True
3,ID000008940,wiley,Atomic force microscopy imaging of novel macro...,polymer_modification_combination,data/fulltext/wiley/ID000008940/ID000008940.pdf,/Users/kevinge/Work/Data Extraction/synth_extr...,True
4,ID000025410,wiley,Reinforced Polymer Blend Membranes with Liposo...,polymer_modification_combination,data/fulltext/wiley/ID000025410/ID000025410.pdf,/Users/kevinge/Work/Data Extraction/synth_extr...,True


In [21]:
# Provider-facing JSON schema. It mirrors ItemList without adding constraints
# such as uniqueItems, so repeated discoveries remain valid.
item_list_response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "item_list",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "description": {"type": "string"},
                            "evidence": {
                                "type": "array",
                                "items": {"type": "string"},
                            },
                        },
                        "required": ["name", "description", "evidence"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["items"],
            "additionalProperties": False,
        },
    },
}


class ComponentExtractionRequest(ExtractionModel):
    uid: str
    markdown_full_text: str


def build_component_extraction_prompt(
    request: ComponentExtractionRequest,
) -> str:
    return USER_TEMPLATE.format(
        task_definition=COMPONENT_TASK.strip(),
        # task_definition=PROPERTY_TASK.strip(),
        # task_definition=PROCESS_TASK.strip(),
        markdown_full_text=request.markdown_full_text,
    )


component_extraction_task = StructuredTask[
    ComponentExtractionRequest,
    ItemList,
](
    name="component_extraction",
    prompt_builder=build_component_extraction_prompt,
    output_model=ItemList,
    response_format=item_list_response_format,
    system_prompt=SYSTEM_PROMPT.strip(),
)

ItemList.model_validate({"items": []})

ItemList(items=[])

In [22]:
max_parallel_requests = 8
extraction_limit: int | None = 10  # Set an integer for a smaller trial run.

selected_paper_records = (
    paper_records
    if extraction_limit is None
    else paper_records[:extraction_limit]
)
print(f"Papers selected for extraction: {len(selected_paper_records):,}")

Papers selected for extraction: 10


In [23]:
async def extract_paper(
    number: int,
    paper: dict[str, Any],
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    uid = str(paper["uid"])
    paper_path = Path(paper["paper_path"])

    try:
        async with semaphore:
            markdown_full_text = await asyncio.to_thread(
                paper_path.read_text,
                encoding="utf-8",
                errors="replace",
            )
            request = ComponentExtractionRequest(
                uid=uid,
                markdown_full_text=markdown_full_text,
            )
            outcome = await agent.arun(
                component_extraction_task,
                request,
            )

        if not isinstance(outcome, StructuredSuccess):
            print(
                f"[{number}/{len(selected_paper_records)}] "
                f"UID: {uid} | {outcome.error_type}: {outcome.message}"
            )
            return {
                **paper,
                "status": "error",
                "result": None,
                "raw_response": outcome.raw_response,
                "reasoning": None,
                "usage": None,
                "error_type": outcome.error_type,
                "error_message": outcome.message,
            }

        validated_result = outcome.result
        usage = outcome.metadata.usage
        usage_data = usage.model_dump() if usage is not None else None
        print(
            f"[{number}/{len(selected_paper_records)}] "
            f"UID: {uid} | items: {len(validated_result.items)}"
        )
        return {
            **paper,
            "status": "ok",
            "result": validated_result,
            "raw_response": outcome.raw_response,
            "reasoning": outcome.metadata.reasoning,
            "usage": usage_data,
            "error_type": None,
            "error_message": None,
        }
    except Exception as exc:
        print(
            f"[{number}/{len(selected_paper_records)}] "
            f"UID: {uid} | {type(exc).__name__}: {exc}"
        )
        return {
            **paper,
            "status": "error",
            "result": None,
            "raw_response": None,
            "reasoning": None,
            "usage": None,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }


async def extract_selected_papers(
    papers: list[dict[str, Any]],
    max_parallel: int = 8,
) -> list[dict[str, Any]]:
    if max_parallel < 1:
        raise ValueError("max_parallel must be at least 1")

    semaphore = asyncio.Semaphore(max_parallel)
    tasks = [
        asyncio.create_task(extract_paper(number, paper, semaphore))
        for number, paper in enumerate(papers, start=1)
    ]
    return await asyncio.gather(*tasks)

In [24]:
# Run the extraction. Nothing is written back to SQLite.
extraction_runs = await extract_selected_papers(
    selected_paper_records,
    max_parallel=max_parallel_requests,
)

# Full records include metadata, raw model text, validated Pydantic output,
# token usage, and any error information for manual diagnosis.
extraction_runs_by_uid = {
    str(run["uid"]): run
    for run in extraction_runs
}
validated_results_by_uid: dict[str, ItemList] = {
    uid: run["result"]
    for uid, run in extraction_runs_by_uid.items()
    if run["result"] is not None
}

extraction_review_df = pd.DataFrame(
    {
        "uid": run["uid"],
        "source": run["source"],
        "category": run["category"],
        "title": run["title"],
        "paper_path": str(run["paper_path"]),
        "status": run["status"],
        "n_items": (
            len(run["result"].items)
            if run["result"] is not None
            else None
        ),
        "items": (
            run["result"].model_dump()["items"]
            if run["result"] is not None
            else None
        ),
        "error_type": run["error_type"],
        "error_message": run["error_message"],
    }
    for run in extraction_runs
)

print(f"Successful: {len(validated_results_by_uid):,}")
print(f"Failed: {len(extraction_runs) - len(validated_results_by_uid):,}")
display(extraction_review_df)

# Example manual inspection:
# uid = next(iter(extraction_runs_by_uid))
# display(extraction_runs_by_uid[uid]["result"])
# print(extraction_runs_by_uid[uid]["raw_response"])

[4/10] UID: ID000008940 | items: 12
[3/10] UID: ID000006962 | items: 14
[1/10] UID: ID000000625 | items: 14
[7/10] UID: ID000033843 | items: 12
[8/10] UID: ID000049572 | items: 19
[5/10] UID: ID000025410 | items: 24
[2/10] UID: ID000006174 | items: 36
[9/10] UID: ID000052685 | items: 24
[10/10] UID: ID000054444 | items: 25
[6/10] UID: ID000028597 | items: 40
Successful: 10
Failed: 0


,uid,source,category,title,paper_path,status,n_items,items,error_type,error_message
0,ID000000625,wiley,polymer_modification_combination,Characterization of bis-(phenoxy)phosphazene p...,/Users/kevinge/Work/Data Extraction/synth_extr...,ok,14,"[{'name': 'PPOP (Orgaflex® ADP 300)', 'descrip...",None,None
1,ID000006174,wiley,polymer_synthesis,Incorporation of adhesion peptides into nonadh...,/Users/kevinge/Work/Data Extraction/synth_extr...,ok,36,"[{'name': 'YRGDS peptide', 'description': 'YRG...",None,None
2,ID000006962,wiley,polymer_modification_combination,Effect of the network structure on thermal and...,/Users/kevinge/Work/Data Extraction/synth_extr...,ok,14,"[{'name': 'Biphenol diglycidyl ether (BP)', 'd...",None,None
3,ID000008940,wiley,polymer_modification_combination,Atomic force microscopy imaging of novel macro...,/Users/kevinge/Work/Data Extraction/synth_extr...,ok,12,"[{'name': 'Atactic polystyrene', 'description'...",None,None
4,ID000025410,wiley,polymer_modification_combination,Reinforced Polymer Blend Membranes with Liposo...,/Users/kevinge/Work/Data Extraction/synth_extr...,ok,24,[{'name': 'Bis(4-chloro-3-sulfophenyl)sulfone ...,None,None
5,ID000028597,wiley,polymer_composite_formulation,Lipid-Polymer Hybrid “Particle-in-Particle” Na...,/Users/kevinge/Work/Data Extraction/synth_extr...,ok,40,"[{'name': 'PLGA-COOH', 'description': 'Poly (D...",None,None
6,ID000033843,wiley,polymer_composite_formulation,Microenvironment-Responsive Injectable Microsp...,/Users/kevinge/Work/Data Extraction/synth_extr...,ok,12,"[{'name': 'Gelatin methacryloyl (GelMA)', 'des...",None,None
7,ID000049572,wiley,polymer_modification_combination,"A Mechanochemically Triggered ""click"" Catalyst",/Users/kevinge/Work/Data Extraction/synth_extr...,ok,19,"[{'name': 'Biscarbene complex I (PIB-based)', ...",None,None
8,ID000052685,wiley,polymer_synthesis,A Triphenylphosphine-Based Microporous Polymer...,/Users/kevinge/Work/Data Extraction/synth_extr...,ok,24,"[{'name': '1,4-dibromobenzene', 'description':...",None,None
9,ID000054444,wiley,polymer_synthesis,“Noninnate” Ring-Opening Polymerization: Palla...,/Users/kevinge/Work/Data Extraction/synth_extr...,ok,25,[{'name': 'Diethyl 2-(phenylethynyl)cyclopropa...,None,None


In [25]:
# for _, row in extraction_review_df[extraction_review_df["category"] == "polymer_synthesis"].iterrows():
for _, row in extraction_review_df.iterrows():
    # if row["uid"] != "ID001100908":
    #     continue
    print(row["uid"])
    print(row["title"])
    print("-"*10)
    for item in row["items"]:
        print("name         :", item["name"])
        print("description  :", item["description"])
        # print("evidence.    :", item["evidence"])
        print(" ")

ID000000625
Characterization of bis-(phenoxy)phosphazene polymers using static secondary ion mass spectrometry
----------
name         : PPOP (Orgaflex® ADP 300)
description  : Poly[bis(phenoxy)phosphazene], with the chemical formula [NP(OC6H5)2]n. This material was used as purchased from Elf Atochem under the trade name Orgaflex® ADP 300. It is a high-molecular-weight polymer with a molecular weight range of 600,000 to 750,000 amu. For sample preparation, it was used as a 5% w/v solution in tetrahydrofuran, requiring warming to completely dissolve the polymer.
 
name         : MeO-PPOP
description  : Poly[bis(4-methoxyphenoxy)phosphazene], with the chemical formula [NP(OC6H4OCH3)2]n. This polymer was prepared according to the published procedure of Allcock by reacting poly(dichlorophosphazene) with the sodium salt of 4-methoxyphenol. For sample preparation, it was used as a 5% w/v solution in 1,4-dioxane, requiring warming to completely dissolve the polymer.
 
name         : EtO2C-PPO

In [57]:
import random
import textwrap

items = []

for _, row in extraction_review_df.iterrows():
    for item in row["items"]:
        items.append(
            {
                "uid": row["uid"],
                "title": row["title"],
                "name": item["name"],
                "description": item["description"],
            }
        )

sample_items = random.Random(423).sample(
    items,
    k=min(5, len(items)),
)

WIDTH = 200

for i, item in enumerate(sample_items, start=1):
    print("-" * WIDTH)
    print(f"uid:   {item['uid']}")
    print(f"title: {item['title']}")
    

    print("name:", end="")
    print(textwrap.fill(
        item["name"],
        width=WIDTH,
        initial_indent="  ",
        subsequent_indent="  ",
    ))

    print("description:" , end="")
    print(textwrap.fill(
        item["description"],
        width=WIDTH,
        initial_indent="  ",
        subsequent_indent="  ",
    ))

    print()

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
uid:   ID000025410
title: Reinforced Polymer Blend Membranes with Liposome-Like Morphology for Polymer Electrolyte Membrane Fuel Cells Operating under Low-Humidity Conditions
name:  Toluene
description:  Toluene is a solvent purchased from Sigma-Aldrich. It is used in azeotropic distillation with water during the polycondensation synthesis of BPSH60, PES-OH60, and SPES40-OH40.

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
uid:   ID000006174
title: Incorporation of adhesion peptides into nonadhesive hydrogels useful for tissue resurfacing
name:  Formaldehyde
description:  4% formaldehyde solution obtained from Fisher Scientific, used to

## Axis-based description distillation

Distill each extracted component description independently along the role, function, chemical-class, and physical-form axes. The original extraction results remain unchanged.

In [26]:
import json


class DistilledItem(ExtractionModel):
    source_item_id: str = Field(
        description="Identifier of the input item, copied exactly."
    )
    distilled_description: str = Field(
        description=(
            "One short axis-specific sentence, or exactly NOT_STATED."
        )
    )


class DistilledItemList(ExtractionModel):
    items: list[DistilledItem] = Field(default_factory=list)


class DistillationInputItem(ExtractionModel):
    source_item_id: str
    name: str
    description: str


class ComponentDistillationRequest(ExtractionModel):
    axis: str
    items: list[DistillationInputItem]


# Keep the provider schema simple: ordering and ID uniqueness are checked
# in Python rather than expressed with unsupported JSON Schema keywords.
distilled_item_list_response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "distilled_item_list",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "source_item_id": {"type": "string"},
                            "distilled_description": {"type": "string"},
                        },
                        "required": [
                            "source_item_id",
                            "distilled_description",
                        ],
                        "additionalProperties": False,
                    },
                },
            },
            "required": ["items"],
            "additionalProperties": False,
        },
    },
}

DistilledItemList.model_validate({"items": []})

DistilledItemList(items=[])

In [27]:
DISTILL_PROMPT = """
You will distill descriptions of items so they can be grouped later by {axis_name}.

Question: {axis_question}
In scope: {axis_in_scope}
Ignore: {axis_ignore}
Source of the answer: {axis_knowledge_rule}

For each item, write one short sentence (5–20 words) that answers the question.

Rules
- Keep the description's own terms where it uses them.
- Leave out the item's name and abbreviation, numbers, amounts, purity,
  supplier, and anything else listed under Ignore.
- If the item has more than one answer (e.g. used in two different ways),
  mention each one in the sentence.
- If the question can't be answered under the source rule above, write
  exactly NOT_STATED.
- Treat each item on its own; don't reuse wording from other items.
- Return one entry for every input item, in the same order, with
  source_item_id copied exactly.

Items:
{items_json}

Return exactly one JSON object with this structure:

{{
  \"items\": [
    {{
      \"source_item_id\": \"The id of the input item, copied exactly.\",
      \"distilled_description\": \"One sentence of 5–20 words answering the question, or NOT_STATED.\"
    }}
  ]
}}

Return only the JSON object, with no text before or after it.
"""


AXIS_CARDS = {
    "role": {
        "axis_name": "role",
        "axis_question": "How is the item used in the experimental procedure?",
        "axis_in_scope": (
            "What the item is added or used for while making, processing, "
            "purifying or characterizing the sample."
        ),
        "axis_ignore": (
            "chemical identity or class, physical form, amounts, concentrations, "
            "purity, molecular weight, supplier, the property it gives the final "
            "material, kinetics or mechanism."
        ),
        "axis_knowledge_rule": (
            "Only what the description states. Do not infer the role from what "
            "the chemical usually does."
        ),
    },
    "function": {
        "axis_name": "function",
        "axis_question": (
            "What effect is the item reported to have on the final material?"
        ),
        "axis_in_scope": (
            "A property or behaviour of the product that the item imparts, "
            "modifies or enables."
        ),
        "axis_ignore": (
            "how it is used in the procedure, chemical identity, physical form, "
            "amounts, purity, supplier."
        ),
        "axis_knowledge_rule": (
            "Only what the description states. Do not infer an effect the "
            "description does not report."
        ),
    },
    "chemical_class": {
        "axis_name": "chemical class",
        "axis_question": "What kind of chemical is the item?",
        "axis_in_scope": (
            "Chemical family or structural class, at the level a chemist would "
            "name it (e.g. functional group, compound class, polymer family)."
        ),
        "axis_ignore": (
            "how it is used, the effect it has, physical form, amounts, purity, "
            "supplier."
        ),
        "axis_knowledge_rule": (
            "Use the description first; you may infer the class from the item's "
            "name when the description does not state it."
        ),
    },
    "physical_form": {
        "axis_name": "physical form",
        "axis_question": "In what physical form is the item used?",
        "axis_in_scope": (
            "State and morphology as used: e.g. particles, fibres, film, powder, "
            "solution, dispersion, liquid, and a size class if stated."
        ),
        "axis_ignore": (
            "how it is used, the effect it has, chemical identity, amounts, "
            "purity, supplier."
        ),
        "axis_knowledge_rule": "Only what the description states.",
    },
}

In [28]:
max_parallel_distill_requests = 8


def distillation_format_warnings(
    result: DistilledItemList,
) -> list[str]:
    warnings: list[str] = []
    for item in result.items:
        text = item.distilled_description.strip()
        if text == "NOT_STATED":
            continue
        word_count = len(text.split())
        if not 5 <= word_count <= 20:
            warnings.append(
                f"{item.source_item_id}: {word_count} words (expected 5–20)"
            )
    return warnings


def build_component_distillation_prompt(
    request: ComponentDistillationRequest,
) -> str:
    if request.axis not in AXIS_CARDS:
        raise ValueError(
            f"Unknown axis {request.axis!r}; choose one of {list(AXIS_CARDS)}"
        )
    items_json = json.dumps(
        [item.model_dump() for item in request.items],
        ensure_ascii=False,
        indent=2,
    )
    return DISTILL_PROMPT.format(
        **AXIS_CARDS[request.axis],
        items_json=items_json,
    )


def validate_component_distillation(
    request: ComponentDistillationRequest,
    result: DistilledItemList,
) -> None:
    expected_ids = [item.source_item_id for item in request.items]
    returned_ids = [item.source_item_id for item in result.items]
    if returned_ids != expected_ids:
        raise ValueError(
            f"Expected item IDs {expected_ids}, got {returned_ids}"
        )


component_distillation_task = StructuredTask[
    ComponentDistillationRequest,
    DistilledItemList,
](
    name="component_distillation",
    prompt_builder=build_component_distillation_prompt,
    output_model=DistilledItemList,
    response_format=distilled_item_list_response_format,
    validate_result=validate_component_distillation,
)


async def distill_paper_axis(
    number: int,
    total: int,
    uid: str,
    original_result: ItemList,
    axis_key: str,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    request = ComponentDistillationRequest(
        axis=axis_key,
        items=[
            DistillationInputItem(
                source_item_id=f"item_{item_number:04d}",
                name=item.name,
                description=item.description,
            )
            for item_number, item in enumerate(
                original_result.items,
                start=1,
            )
        ],
    )

    if not request.items:
        return {
            "uid": uid,
            "axis": axis_key,
            "status": "ok",
            "result": DistilledItemList(items=[]),
            "raw_response": None,
            "reasoning": None,
            "usage": None,
            "format_warnings": [],
            "error_type": None,
            "error_message": None,
        }

    try:
        async with semaphore:
            outcome = await agent.arun(
                component_distillation_task,
                request,
            )

        if not isinstance(outcome, StructuredSuccess):
            print(
                f"[{number}/{total}] UID: {uid} | axis: {axis_key} | "
                f"{outcome.error_type}: {outcome.message}"
            )
            return {
                "uid": uid,
                "axis": axis_key,
                "status": "error",
                "result": None,
                "raw_response": outcome.raw_response,
                "reasoning": None,
                "usage": None,
                "format_warnings": [],
                "error_type": outcome.error_type,
                "error_message": outcome.message,
            }

        validated_result = outcome.result
        usage = outcome.metadata.usage
        usage_data = usage.model_dump() if usage is not None else None
        format_warnings = distillation_format_warnings(validated_result)
        print(
            f"[{number}/{total}] UID: {uid} | axis: {axis_key} | "
            f"items: {len(validated_result.items)} | "
            f"warnings: {len(format_warnings)}"
        )
        return {
            "uid": uid,
            "axis": axis_key,
            "status": "ok",
            "result": validated_result,
            "raw_response": outcome.raw_response,
            "reasoning": outcome.metadata.reasoning,
            "usage": usage_data,
            "format_warnings": format_warnings,
            "error_type": None,
            "error_message": None,
        }
    except Exception as exc:
        print(
            f"[{number}/{total}] UID: {uid} | axis: {axis_key} | "
            f"{type(exc).__name__}: {exc}"
        )
        return {
            "uid": uid,
            "axis": axis_key,
            "status": "error",
            "result": None,
            "raw_response": None,
            "reasoning": None,
            "usage": None,
            "format_warnings": [],
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }


async def distill_component_axis(
    results_by_uid: dict[str, ItemList],
    axis_key: str,
    max_parallel: int = 8,
) -> list[dict[str, Any]]:
    if max_parallel < 1:
        raise ValueError("max_parallel must be at least 1")
    if axis_key not in AXIS_CARDS:
        raise ValueError(
            f"Unknown axis {axis_key!r}; choose one of {list(AXIS_CARDS)}"
        )

    jobs = list(results_by_uid.items())
    semaphore = asyncio.Semaphore(max_parallel)
    total = len(jobs)
    tasks = [
        asyncio.create_task(
            distill_paper_axis(
                number=number,
                total=total,
                uid=uid,
                original_result=original_result,
                axis_key=axis_key,
                semaphore=semaphore,
            )
        )
        for number, (uid, original_result) in enumerate(jobs, start=1)
    ]
    return await asyncio.gather(*tasks)

In [29]:
# Select exactly one component axis for this run.
# Available values: role, function, chemical_class, physical_form.
distillation_axis = "role"

# Run one distillation request per successfully extracted paper.
# No values are written to SQLite.
distillation_runs = await distill_component_axis(
    validated_results_by_uid,
    axis_key=distillation_axis,
    max_parallel=max_parallel_distill_requests,
)

distillation_runs_by_uid: dict[str, dict[str, Any]] = {
    run["uid"]: run for run in distillation_runs
}
distilled_results_by_uid: dict[str, DistilledItemList] = {
    uid: run["result"]
    for uid, run in distillation_runs_by_uid.items()
    if run["result"] is not None
}

successful_distillations = sum(
    run["status"] == "ok" for run in distillation_runs
)
print(f"Axis: {distillation_axis}")
print(f"Successful paper requests: {successful_distillations:,}")
print(f"Failed paper requests: {len(distillation_runs) - successful_distillations:,}")

[7/10] UID: ID000033843 | axis: role | items: 12 | warnings: 0
[4/10] UID: ID000008940 | axis: role | items: 12 | warnings: 0
[3/10] UID: ID000006962 | axis: role | items: 14 | warnings: 0
[1/10] UID: ID000000625 | axis: role | items: 14 | warnings: 0
[8/10] UID: ID000049572 | axis: role | items: 19 | warnings: 0
[5/10] UID: ID000025410 | axis: role | items: 24 | warnings: 0
[9/10] UID: ID000052685 | axis: role | items: 24 | warnings: 0
[10/10] UID: ID000054444 | axis: role | items: 25 | warnings: 0
[2/10] UID: ID000006174 | axis: role | items: 36 | warnings: 0
[6/10] UID: ID000028597 | axis: role | items: 40 | warnings: 0
Axis: role
Successful paper requests: 10
Failed paper requests: 0


In [30]:
# One row per original component for the selected axis.
distillation_review_rows: list[dict[str, Any]] = []

for uid, original_result in validated_results_by_uid.items():
    paper_metadata = extraction_runs_by_uid[uid]
    distillation_run = distillation_runs_by_uid.get(uid)
    distilled_by_item_id = (
        {
            item.source_item_id: item.distilled_description
            for item in distillation_run["result"].items
        }
        if distillation_run is not None
        and distillation_run["result"] is not None
        else {}
    )

    for item_number, original_item in enumerate(
        original_result.items, start=1
    ):
        source_item_id = f"item_{item_number:04d}"
        row = {
            "uid": uid,
            "source": paper_metadata["source"],
            "category": paper_metadata["category"],
            "title": paper_metadata["title"],
            "source_item_id": source_item_id,
            "name": original_item.name,
            "original_description": original_item.description,
            "axis": distillation_axis,
            "distilled_description": distilled_by_item_id.get(
                source_item_id
            ),
            "evidence": original_item.evidence,
        }
        distillation_review_rows.append(row)

distillation_review_df = pd.DataFrame(distillation_review_rows)
distillation_warning_df = pd.DataFrame(
    {
        "uid": run["uid"],
        "axis": run["axis"],
        "warning": warning,
    }
    for run in distillation_runs
    for warning in run["format_warnings"]
)

display(distillation_review_df)
if not distillation_warning_df.empty:
    print("Descriptions outside the requested 5–20-word range:")
    display(distillation_warning_df)

,uid,source,category,title,source_item_id,name,original_description,axis,distilled_description,evidence
0,ID000000625,wiley,polymer_modification_combination,Characterization of bis-(phenoxy)phosphazene p...,item_0001,PPOP (Orgaflex® ADP 300),"Poly[bis(phenoxy)phosphazene], with the chemic...",role,Used as a solution in tetrahydrofuran for samp...,"[PPOP: poly[bis(phenoxy)phosphazene], [NP(OC6H..."
1,ID000000625,wiley,polymer_modification_combination,Characterization of bis-(phenoxy)phosphazene p...,item_0002,MeO-PPOP,"Poly[bis(4-methoxyphenoxy)phosphazene], with t...",role,"Used as a solution in 1,4-dioxane for sample p...",[MeO-PPOP: poly[bis(4-methoxyphenoxy)phosphaze...
2,ID000000625,wiley,polymer_modification_combination,Characterization of bis-(phenoxy)phosphazene p...,item_0003,EtO2C-PPOP,Poly[bis(ethyl 4-carboxylatophenoxy)phosphazen...,role,Used as a solution in tetrahydrofuran for samp...,[EtO2C-PPOP: poly[bis(ethyl 4-carboxylatopheno...
3,ID000000625,wiley,polymer_modification_combination,Characterization of bis-(phenoxy)phosphazene p...,item_0004,HO2C-PPOP,"Poly[bis(4-carboxylatophenoxy)phosphazene], wi...",role,Dissolved in water with ammonium hydroxide for...,[HO2C-PPOP: poly[bis(4-carboxylatophenoxy)phos...
4,ID000000625,wiley,polymer_modification_combination,Characterization of bis-(phenoxy)phosphazene p...,item_0005,Poly(dichlorophosphazene),A polymer precursor consisting of a backbone o...,role,Used as starting material for synthesizing pol...,[This allows synthesis of one polymer precurso...
...,...,...,...,...,...,...,...,...,...,...
215,ID000054444,wiley,polymer_synthesis,“Noninnate” Ring-Opening Polymerization: Palla...,item_0021,DBU,"1,8-Diazabicyclo[5.4.0]undec-7-ene (DBU) is an...",role,Used as organic base that produced only oligom...,"[In contrast, an organic base, 1,8-diazabicycl..."
216,ID000054444,wiley,polymer_synthesis,“Noninnate” Ring-Opening Polymerization: Palla...,item_0022,Acetonitrile,Acetonitrile (MeCN) is a solvent that serves a...,role,Used as viable alternative solvent for the pol...,[Solvent screening revealed acetonitrile and t...
217,ID000054444,wiley,polymer_synthesis,“Noninnate” Ring-Opening Polymerization: Palla...,item_0023,Toluene,Toluene is a solvent that serves as a viable a...,role,Used as viable alternative solvent for polymer...,[Solvent screening revealed acetonitrile and t...
218,ID000054444,wiley,polymer_synthesis,“Noninnate” Ring-Opening Polymerization: Palla...,item_0024,DMSO,Dimethyl sulfoxide (DMSO) was tested as a solv...,role,Tested as solvent for polymerization but did n...,"[Entry 13: NaOtBu, DPEPhos, in DMSO, Yield -]"


In [31]:
# for _, row in extraction_review_df[extraction_review_df["category"] == "polymer_synthesis"].iterrows():
for _, row in distillation_review_df.iterrows():
    # if row["uid"] != "ID001100908":
    #     continue
    print(row["uid"])
    print(row["title"])
    print("-"*10)
    # for item in row["items"]:
    print("name         :", row["name"])
    print("description  :", row["original_description"])
    print("distilled    :", row["distilled_description"])
    print(" ")

ID000000625
Characterization of bis-(phenoxy)phosphazene polymers using static secondary ion mass spectrometry
----------
name         : PPOP (Orgaflex® ADP 300)
description  : Poly[bis(phenoxy)phosphazene], with the chemical formula [NP(OC6H5)2]n. This material was used as purchased from Elf Atochem under the trade name Orgaflex® ADP 300. It is a high-molecular-weight polymer with a molecular weight range of 600,000 to 750,000 amu. For sample preparation, it was used as a 5% w/v solution in tetrahydrofuran, requiring warming to completely dissolve the polymer.
distilled    : Used as a solution in tetrahydrofuran for sample preparation.
 
ID000000625
Characterization of bis-(phenoxy)phosphazene polymers using static secondary ion mass spectrometry
----------
name         : MeO-PPOP
description  : Poly[bis(4-methoxyphenoxy)phosphazene], with the chemical formula [NP(OC6H4OCH3)2]n. This polymer was prepared according to the published procedure of Allcock by reacting poly(dichlorophosphaz

In [59]:
import random
import textwrap

items = []

for _, row in distillation_review_df.iterrows():
    items.append(
        {
            "uid": row["uid"],
            "title": row["title"],
            "name": row["name"],
            "original_description": row["original_description"],
            "distilled_description": row["distilled_description"],
        }
    )

sample_items = random.Random(453).sample(
    items,
    k=min(5, len(items)),
)

WIDTH = 200

for i, item in enumerate(sample_items, start=1):
    print("-" * WIDTH)
    print(f"uid:   {item['uid']}")
    print(f"title: {item['title']}")

    print("name:", end="")
    print(textwrap.fill(
        item["name"],
        width=WIDTH,
        initial_indent="  ",
        subsequent_indent="  ",
    ))

    print("original:", end="")
    print(textwrap.fill(
        item["original_description"],
        width=WIDTH,
        initial_indent="  ",
        subsequent_indent="  ",
    ))

    print("distilled:", end="")
    print(textwrap.fill(
        item["distilled_description"],
        width=WIDTH,
        initial_indent="  ",
        subsequent_indent="  ",
    ))

    print()

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
uid:   ID000006962
title: Effect of the network structure on thermal and mechanical properties of mesogenic epoxy resin cured with aromatic amine
name:  Formaldehyde
original:  Formaldehyde is a reactant used in the synthesis of catechol novolak. It is mixed with catechol in a 1:2 molar ratio.
distilled:  Used as a reactant in the synthesis of catechol novolak.

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
uid:   ID000028597
title: Lipid-Polymer Hybrid “Particle-in-Particle” Nanostructure Gene Delivery Platform Explored for Lyophilizable DNA and mRNA COVID-19 Vaccines
name:  1,4-Butanediol diacrylate
original:  A monomer obtained from 

Finding candidate concepts

In [43]:
class ConceptCandidate(ExtractionModel):
    name: str
    definition: str
    include: str
    exclude: str
    supporting_item_ids: list[str] = Field(default_factory=list)
    near_miss_item_ids: list[str] = Field(default_factory=list)


class ConceptCandidateList(ExtractionModel):
    concepts: list[ConceptCandidate] = Field(default_factory=list)


class ConceptDiscoveryItem(ExtractionModel):
    id: str
    text: str


class ConceptDiscoveryRequest(ExtractionModel):
    axis: str
    max_concepts: int
    items: list[ConceptDiscoveryItem]


concept_candidate_response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "concept_candidate_list",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "concepts": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "definition": {"type": "string"},
                            "include": {"type": "string"},
                            "exclude": {"type": "string"},
                            "supporting_item_ids": {
                                "type": "array",
                                "items": {"type": "string"},
                            },
                            "near_miss_item_ids": {
                                "type": "array",
                                "items": {"type": "string"},
                            },
                        },
                        "required": [
                            "name",
                            "definition",
                            "include",
                            "exclude",
                            "supporting_item_ids",
                            "near_miss_item_ids",
                        ],
                        "additionalProperties": False,
                    },
                },
            },
            "required": ["concepts"],
            "additionalProperties": False,
        },
    },
}


CONCEPT_DISCOVERY_PROMPT = """
You are finding the recurring kinds of items along one axis.

Axis: {axis_name}
Question: {axis_question}
In scope: {axis_in_scope}
Ignore: {axis_ignore}

Below are short descriptions of items from many different papers. Each one
already answers the question above. Propose up to {max_concepts} concepts.
Each concept is one recurring kind of answer to the question.

Rules
- Every concept must be an answer to the question. Do not group items by
  anything listed under Ignore.
- Prefer the MOST SPECIFIC kind that at least 2 items share. If several
  specific kinds clearly belong together, you may also propose their
  broader parent as a separate concept.
- Each concept must fit at least 2 items in this list. Never propose a
  concept supported by only 1 item.
- You do not need to cover every item. Leave out items that share no kind
  with any other item.
- An item may describe more than one use; it can support more than one concept.
- Check each supporting id against your own include and exclude text. List
  an id only if its description meets the include condition as written.
- Pay attention to direction: an item that dissolves something is different
  from an item that is dissolved; an item that reacts with a polymer is
  different from the polymer being modified.
- Pay attention to the stage of the work when it changes what the item does.
- The name must not contain \"or\" or \"and\". If you need \"or\",
  propose two concepts.
- No catch-all concepts such as \"other\", \"miscellaneous\", or
  \"general reagent\".
- A concept must be testable on a single description alone.
- name: 2-4 words, as a researcher in the field would say it.
- definition: one sentence stating what makes an item this kind.
- include: the conditions an item's description must meet to belong.
- exclude: what does not belong, especially the closest kind it could be
  confused with.
- near_miss_item_ids: ids in this list that look similar but do not belong
  (may be empty).

Items:
{items_json}

Return exactly one JSON object with this structure:

{{
  \"concepts\": [
    {{
      \"name\": \"Short name of the kind (2-4 words).\",
      \"definition\": \"One sentence stating what makes an item this kind.\",
      \"include\": \"Conditions an item's description must meet to belong.\",
      \"exclude\": \"What does not belong, including the closest confusable kind.\",
      \"supporting_item_ids\": [\"ids from this list that clearly fit\"],
      \"near_miss_item_ids\": [\"similar ids from this list that do not fit\"]
    }}
  ]
}}

Return only the JSON object, with no text before or after it.
"""


def build_concept_discovery_prompt(
    request: ConceptDiscoveryRequest,
) -> str:
    if request.axis not in AXIS_CARDS:
        raise ValueError(
            f"Unknown axis {request.axis!r}; choose one of {list(AXIS_CARDS)}"
        )
    items_json = json.dumps(
        [item.model_dump() for item in request.items],
        ensure_ascii=False,
        indent=2,
    )
    return CONCEPT_DISCOVERY_PROMPT.format(
        **AXIS_CARDS[request.axis],
        max_concepts=request.max_concepts,
        items_json=items_json,
    )


def concept_candidate_rejection_reasons(
    request: ConceptDiscoveryRequest,
    concept: ConceptCandidate,
    concept_number: int,
) -> list[str]:
    reasons: list[str] = []
    if concept_number > request.max_concepts:
        reasons.append(
            f"concept exceeds the maximum of {request.max_concepts}"
        )
    valid_ids = {item.id for item in request.items}
    supporting_ids = concept.supporting_item_ids
    near_miss_ids = concept.near_miss_item_ids
    if len(supporting_ids) != len(set(supporting_ids)):
        reasons.append("duplicate supporting item IDs")
    if len(set(supporting_ids)) < 2:
        reasons.append("fewer than 2 distinct supporting items")
    if len(near_miss_ids) != len(set(near_miss_ids)):
        reasons.append("duplicate near-miss item IDs")
    unknown_ids = (set(supporting_ids) | set(near_miss_ids)) - valid_ids
    if unknown_ids:
        reasons.append(f"unknown item IDs: {sorted(unknown_ids)}")
    overlap = set(supporting_ids) & set(near_miss_ids)
    if overlap:
        reasons.append(
            f"IDs used as both supporting and near misses: {sorted(overlap)}"
        )
    for field_name in ("name", "definition", "include", "exclude"):
        if not getattr(concept, field_name).strip():
            reasons.append(f"empty {field_name}")
    return reasons


concept_discovery_task = StructuredTask[
    ConceptDiscoveryRequest,
    ConceptCandidateList,
](
    name="concept_discovery",
    prompt_builder=build_concept_discovery_prompt,
    output_model=ConceptCandidateList,
    response_format=concept_candidate_response_format,
)

In [44]:
concept_axis = distillation_axis
concept_batch_size = 50
concept_max_per_batch = 8
concept_random_seed = 42
max_parallel_concept_requests = 8

if concept_axis not in AXIS_CARDS:
    raise ValueError(
        f"Unknown axis {concept_axis!r}; choose one of {list(AXIS_CARDS)}"
    )
if concept_batch_size < 2:
    raise ValueError("concept_batch_size must be at least 2")
if concept_max_per_batch < 1:
    raise ValueError("concept_max_per_batch must be at least 1")

concept_source_df = distillation_review_df.loc[
    (distillation_review_df["axis"] == concept_axis)
    & distillation_review_df["distilled_description"].notna()
].copy()
concept_source_df["distilled_description"] = concept_source_df[
    "distilled_description"
].str.strip()
not_stated_count = int(
    (concept_source_df["distilled_description"] == "NOT_STATED").sum()
)
concept_source_df = concept_source_df.loc[
    (concept_source_df["distilled_description"] != "")
    & (concept_source_df["distilled_description"] != "NOT_STATED")
].copy()
if concept_source_df.empty:
    raise ValueError(
        f"No usable distilled descriptions for axis {concept_axis!r}"
    )

# Shuffle once, then take consecutive non-overlapping batches.
concept_items_df = concept_source_df.sample(
    frac=1.0,
    random_state=concept_random_seed,
).reset_index(drop=True)
concept_items_df.insert(
    0,
    "concept_item_id",
    [f"i{number:06d}" for number in range(1, len(concept_items_df) + 1)],
)

concept_batches: list[ConceptDiscoveryRequest] = []
for start in range(0, len(concept_items_df), concept_batch_size):
    batch_df = concept_items_df.iloc[start : start + concept_batch_size]
    concept_batches.append(
        ConceptDiscoveryRequest(
            axis=concept_axis,
            max_concepts=concept_max_per_batch,
            items=[
                ConceptDiscoveryItem(
                    id=row.concept_item_id,
                    text=row.distilled_description,
                )
                for row in batch_df.itertuples(index=False)
            ],
        )
    )

batched_item_ids = [
    item.id for batch in concept_batches for item in batch.items
]
expected_item_ids = concept_items_df["concept_item_id"].tolist()
if batched_item_ids != expected_item_ids:
    raise AssertionError("Concept batches do not preserve every item exactly once")

concept_item_lookup_df = concept_items_df[[
    "concept_item_id",
    "uid",
    "source_item_id",
    "name",
    "distilled_description",
]].copy()
concept_item_lookup_by_id = concept_item_lookup_df.set_index(
    "concept_item_id"
).to_dict(orient="index")

print(f"Axis: {concept_axis}")
print(f"Usable descriptions: {len(concept_items_df):,}")
print(f"Excluded NOT_STATED descriptions: {not_stated_count:,}")
print(f"Batch size: {concept_batch_size:,}")
print(f"Batches: {len(concept_batches):,}")
display(concept_item_lookup_df.head())

Axis: role
Usable descriptions: 220
Excluded NOT_STATED descriptions: 0
Batch size: 50
Batches: 5


,concept_item_id,uid,source_item_id,name,distilled_description
0,i000001,ID000028597,item_0033,DMG-PEG,Used as a PEG-lipid component in the formulati...
1,i000002,ID000033843,item_0009,Span-80,Used as a surfactant in the oil phase during e...
2,i000003,ID000025410,item_0018,Platinum on carbon (Pt/C),Used as the catalyst in the catalyst slurry fo...
3,i000004,ID000052685,item_0010,"N,N-Dimethylformamide",Used as a solvent in the cross-coupling polyme...
4,i000005,ID000006174,item_0002,YRDGS peptide,"Used as inactive control peptide, functionaliz..."


In [45]:
async def discover_concepts_in_batch(
    batch_number: int,
    total_batches: int,
    request: ConceptDiscoveryRequest,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    async with semaphore:
        outcome = await agent.arun(concept_discovery_task, request)

    if not isinstance(outcome, StructuredSuccess):
        print(
            f"[{batch_number}/{total_batches}] concept batch | "
            f"{outcome.error_type}: {outcome.message}"
        )
        return {
            "batch_number": batch_number,
            "status": "error",
            "request": request,
            "result": None,
            "unfiltered_result": None,
            "rejected_concepts": [],
            "raw_response": outcome.raw_response,
            "usage": None,
            "error_type": outcome.error_type,
            "error_message": outcome.message,
        }

    accepted_concepts: list[ConceptCandidate] = []
    rejected_concepts: list[dict[str, Any]] = []
    for concept_number, concept in enumerate(
        outcome.result.concepts,
        start=1,
    ):
        rejection_reasons = concept_candidate_rejection_reasons(
            request=request,
            concept=concept,
            concept_number=concept_number,
        )
        if rejection_reasons:
            rejected_concepts.append(
                {
                    "concept_number": concept_number,
                    "concept": concept,
                    "reasons": rejection_reasons,
                }
            )
        else:
            accepted_concepts.append(concept)

    filtered_result = ConceptCandidateList(concepts=accepted_concepts)
    usage = outcome.metadata.usage
    print(
        f"[{batch_number}/{total_batches}] concept batch | "
        f"items: {len(request.items)} | "
        f"returned: {len(outcome.result.concepts)} | "
        f"accepted: {len(accepted_concepts)} | "
        f"rejected: {len(rejected_concepts)}"
    )
    return {
        "batch_number": batch_number,
        "status": "ok",
        "request": request,
        "result": filtered_result,
        "unfiltered_result": outcome.result,
        "rejected_concepts": rejected_concepts,
        "raw_response": outcome.raw_response,
        "usage": usage.model_dump() if usage is not None else None,
        "error_type": None,
        "error_message": None,
    }


async def discover_concepts_in_all_batches(
    batches: list[ConceptDiscoveryRequest],
    max_parallel: int = 8,
) -> list[dict[str, Any]]:
    if max_parallel < 1:
        raise ValueError("max_parallel must be at least 1")
    semaphore = asyncio.Semaphore(max_parallel)
    total_batches = len(batches)
    tasks = [
        asyncio.create_task(
            discover_concepts_in_batch(
                batch_number=batch_number,
                total_batches=total_batches,
                request=request,
                semaphore=semaphore,
            )
        )
        for batch_number, request in enumerate(batches, start=1)
    ]
    return await asyncio.gather(*tasks)


concept_discovery_runs = await discover_concepts_in_all_batches(
    concept_batches,
    max_parallel=max_parallel_concept_requests,
)

[4/5] concept batch | items: 50 | returned: 8 | accepted: 4 | rejected: 4
[5/5] concept batch | items: 20 | returned: 8 | accepted: 4 | rejected: 4
[2/5] concept batch | items: 50 | returned: 8 | accepted: 8 | rejected: 0
[1/5] concept batch | items: 50 | returned: 8 | accepted: 8 | rejected: 0
[3/5] concept batch | items: 50 | returned: 8 | accepted: 7 | rejected: 1


In [46]:
concept_discovery_runs_by_batch = {
    run["batch_number"]: run for run in concept_discovery_runs
}
concept_candidates: list[ConceptCandidate] = [
    concept
    for run in concept_discovery_runs
    if run["result"] is not None
    for concept in run["result"].concepts
]
concept_candidate_rows = [
    {
        "batch_number": run["batch_number"],
        "axis": concept_axis,
        **concept.model_dump(),
        "supporting_items": [
            concept_item_lookup_by_id[item_id]
            for item_id in concept.supporting_item_ids
        ],
        "near_miss_items": [
            concept_item_lookup_by_id[item_id]
            for item_id in concept.near_miss_item_ids
        ],
    }
    for run in concept_discovery_runs
    if run["result"] is not None
    for concept in run["result"].concepts
]
concept_candidates_df = pd.DataFrame(concept_candidate_rows)
rejected_concept_rows = [
    {
        "batch_number": run["batch_number"],
        "axis": concept_axis,
        "concept_number": rejected["concept_number"],
        **rejected["concept"].model_dump(),
        "rejection_reasons": rejected["reasons"],
    }
    for run in concept_discovery_runs
    for rejected in run["rejected_concepts"]
]
rejected_concepts_df = pd.DataFrame(rejected_concept_rows)
failed_concept_batches_df = pd.DataFrame(
    {
        "batch_number": run["batch_number"],
        "error_type": run["error_type"],
        "error_message": run["error_message"],
    }
    for run in concept_discovery_runs
    if run["status"] == "error"
)

print(f"Candidate concepts: {len(concept_candidates):,}")
print(f"Rejected individual concepts: {len(rejected_concepts_df):,}")
print(f"Successful batches: {len(concept_discovery_runs) - len(failed_concept_batches_df):,}")
print(f"Failed batches: {len(failed_concept_batches_df):,}")
display(concept_candidates_df)
if not rejected_concepts_df.empty:
    display(rejected_concepts_df)
if not failed_concept_batches_df.empty:
    display(failed_concept_batches_df)

Candidate concepts: 31
Rejected individual concepts: 9
Successful batches: 5
Failed batches: 0


,batch_number,axis,name,definition,include,exclude,supporting_item_ids,near_miss_item_ids,supporting_items,near_miss_items
0,1,role,Polymerization Solvent,The item is used as a solvent medium in which ...,Description states the item is used as a solve...,"Items used as solvents for purification, preci...","[i000004, i000016, i000046]","[i000014, i000035, i000026, i000043]","[{'uid': 'ID000052685', 'source_item_id': 'ite...","[{'uid': 'ID000000625', 'source_item_id': 'ite..."
1,1,role,Precipitation Solvent,The item is used to precipitate a polymer or c...,Description states the item is used for precip...,Items used as solvents for the reaction itself...,"[i000025, i000026, i000049]","[i000012, i000004, i000016]","[{'uid': 'ID000028597', 'source_item_id': 'ite...","[{'uid': 'ID000006174', 'source_item_id': 'ite..."
2,1,role,Reaction Catalyst,The item acts as a catalyst to accelerate a ch...,Description explicitly identifies the item as ...,Items that are reagents consumed in the reacti...,"[i000003, i000015, i000020, i000030, i000034]","[i000031, i000033, i000036, i000047]","[{'uid': 'ID000025410', 'source_item_id': 'ite...","[{'uid': 'ID000028597', 'source_item_id': 'ite..."
3,1,role,Enzyme Catalyst,The item is an enzyme used to catalyze a speci...,Description identifies the item as an enzyme (...,Small molecule catalysts or reagents that are ...,"[i000031, i000033, i000036, i000047]","[i000003, i000030, i000034]","[{'uid': 'ID000028597', 'source_item_id': 'ite...","[{'uid': 'ID000025410', 'source_item_id': 'ite..."
4,1,role,Base Reagent,The item is used as a base to deprotonate a su...,Description states the item is used as a base ...,"Items used as catalysts, solvents, or reactant...","[i000013, i000019, i000021, i000050]","[i000007, i000041]","[{'uid': 'ID000052685', 'source_item_id': 'ite...","[{'uid': 'ID000052685', 'source_item_id': 'ite..."
5,1,role,Hydrogel Network Component,The item is incorporated as a structural or fu...,Description states the item serves as a compon...,"Items used to rinse, stain, or characterize hy...","[i000010, i000018, i000039, i000048]","[i000012, i000022, i000027]","[{'uid': 'ID000033843', 'source_item_id': 'ite...","[{'uid': 'ID000006174', 'source_item_id': 'ite..."
6,1,role,Polymer Synthesis Monomer,The item is a monomer or starting material use...,"Description states the item is a monomer, diol...","Items used as solvents, catalysts, or ligands ...","[i000011, i000040, i000045]","[i000006, i000007, i000008]","[{'uid': 'ID000028597', 'source_item_id': 'ite...","[{'uid': 'ID000028597', 'source_item_id': 'ite..."
7,1,role,Microscopy Staining Agent,The item is used to stain or contrast samples ...,Description states the item is used as a stain...,"Items used as substrates for deposition, or so...","[i000022, i000027]","[i000009, i000014, i000035]","[{'uid': 'ID000025410', 'source_item_id': 'ite...","[{'uid': 'ID000008940', 'source_item_id': 'ite..."
8,2,role,Polymerization Catalyst,An item used to accelerate or enable the chemi...,The description explicitly states the item is ...,"Items used as monomers, initiators, or bases f...","[i000082, i000096]","[i000059, i000071, i000088]","[{'uid': 'ID000052685', 'source_item_id': 'ite...","[{'uid': 'ID000054444', 'source_item_id': 'ite..."
9,2,role,Polymerization Monomer,An item that serves as the fundamental buildin...,The description identifies the item as a monom...,"Items used as solvents, catalysts, initiators,...","[i000066, i000083, i000094]","[i000065, i000060]","[{'uid': 'ID000052685', 'source_item_id': 'ite...","[{'uid': 'ID000049572', 'source_item_id': 'ite..."


,batch_number,axis,concept_number,name,definition,include,exclude,supporting_item_ids,near_miss_item_ids,rejection_reasons
0,3,role,6,Precipitation agent,The item is used to induce the precipitation o...,Description states the item is used to precipi...,Items used as solvents; items used for washing...,[i000129],"[i000109, i000125]",[fewer than 2 distinct supporting items]
1,4,role,5,Polymer Precipitation Solvent,The item is added to a reaction mixture to cau...,Description states the item is used for precip...,Solvents used to dissolve the polymer; solvent...,[i000164],"[i000180, i000151]",[fewer than 2 distinct supporting items]
2,4,role,6,Reaction Quenching Agent,The item is added to terminate a chemical reac...,Description states the item is used to quench ...,Items used to initiate reactions; items used t...,[i000166],"[i000175, i000189]",[fewer than 2 distinct supporting items]
3,4,role,7,Membrane Compatibilizer,The item is added to a polymer blend to improv...,Description states the item acts as a compatib...,Items that are the primary matrix or proton-co...,[i000163],"[i000167, i000189]",[fewer than 2 distinct supporting items]
4,4,role,8,Cell Culture Supplement,The item is added to cell culture media to sup...,Description states the item is used as a suppl...,Items used for sterilization; items used as an...,[i000160],"[i000197, i000158]",[fewer than 2 distinct supporting items]
5,5,role,2,Characterization Solvent,The item serves as the mobile phase or dissolu...,Description states the item is used as a solve...,Items used as solvents for synthesis or reacti...,[i000212],"[i000204, i000211]",[fewer than 2 distinct supporting items]
6,5,role,3,Polymerization Initiator,The item triggers or controls the start and ki...,Description states the item is used as an init...,"Items that are monomers, solvents, or ligands....",[i000209],"[i000205, i000220]",[fewer than 2 distinct supporting items]
7,5,role,4,Chromatography Stationary Phase,The item acts as the fixed phase in a column f...,Description states the item is used as the sta...,Items used as mobile phases (solvents). Items ...,[i000210],[i000212],[fewer than 2 distinct supporting items]
8,5,role,6,Curing Agent,"The item is used to harden, crosslink, or fina...","Description states the item is used to cure, c...",Items used as solvents for crosslinking. Items...,[i000201],[i000204],[fewer than 2 distinct supporting items]
